# Using finance_db in the fundability methodology

Walks the fundability logic (`../../methodology.md`) driven by the `finance_db` tables, to show what each table contributes. The fixture feeds three of the four layers:

- **ACTION** (what is being funded) → `actions.csv`
- **FINANCE / supply** (which money is reachable) → `opportunities.csv`
- **PROJECTS evidence** (precedent + benchmark) → `projects.csv` + `project_funding.csv`

The fourth layer, **CITY** (financial autonomy and delivery capacity), is **not** in finance_db — it comes from the SINIM municipal review. Here it is supplied as a small set of illustrative comuna profiles so the full logic can run end to end.

In [1]:
import pandas as pd, json
opportunities=pd.read_csv("opportunities.csv"); projects=pd.read_csv("projects.csv")
project_funding=pd.read_csv("project_funding.csv"); actions=pd.read_csv("actions.csv")
print("loaded:", {k:len(v) for k,v in {"actions":actions,"opportunities":opportunities,"projects":projects,"project_funding":project_funding}.items()})

loaded: {'actions': 102, 'opportunities': 100, 'projects': 11310, 'project_funding': 11322}


## Layer 1 — ACTION: demand attributes (from `actions.csv`)

Two action demands drive the route. **Capital intensity** loads on the city's *financial autonomy*; **formulation demand** (derived from the archetype) loads on the city's *delivery capacity*. The benchmark profile (timeline, cost) rides along as context.

In [2]:
FORM_DEMAND={"regulatory":0.2,"planning":0.5,"program":0.5,"financial":0.5,"infrastructure":0.8}
a=actions.copy()
a["formulation_demand"]=a["archetype"].map(FORM_DEMAND).fillna(0.5)
a.loc[(a.archetype=="infrastructure")&(a.capital_demand>=0.8),"formulation_demand"]=0.9
a[["action_id","action_name","sector","archetype","capital_demand","formulation_demand",
   "bench_duration_median_months","bench_cost_median_MMCLP","bench_n_projects"]].head(6)

,action_id,action_name,sector,archetype,capital_demand,formulation_demand,bench_duration_median_months,bench_cost_median_MMCLP,bench_n_projects
0,ipcc_0053,"Implement afforestation, reforestation, and fo...",afolu,infrastructure,0.8,0.9,30.0,402.9,24
1,ipcc_0064,Reduce methane emissions from enteric fermenta...,afolu,infrastructure,0.5,0.8,24.0,214.5,19
2,ipcc_0068,Pilot Bioenergy with Carbon Capture in Partner...,afolu,program,0.8,0.5,36.0,386.9,14
3,ipcc_0056,Reduce and prevent degradation and conversion ...,afolu,infrastructure,0.5,0.8,24.0,493.3,6
4,ipcc_0069,Promote sustainable and healthy diets,afolu,program,0.5,0.5,29.5,666.9,4
5,icare_0122,Establish eco-parks.,afolu,infrastructure,0.5,0.8,24.0,297.0,37


## Layer 2 — FINANCE / supply: which money is reachable (from `opportunities.csv`)

For an action's sector, resolve the funds a municipality can actually use: sector match, municipality-eligible, and the access route (direct application versus harder/gated). This yields the **supply** and **access-fit** facets. A *sector-specific* municipal fund is the honest signal; broad cross-sector funds match everything and are tracked separately.

In [3]:
opportunities["secs"]=opportunities["gpc_sectors"].map(lambda s: json.loads(s) if str(s).startswith("[") else [str(s)])
opportunities["is_muni"]=opportunities["eligible_actor"].str.lower().str.contains("municipalit|regional",na=False)
opportunities["is_direct"]=opportunities["access_pathway"].str.lower().str.contains("direct",na=False)

def supply_signal(sector):
    muni=opportunities[opportunities.is_muni]
    specific=muni[muni.secs.apply(lambda S: sector in S)]
    broad   =muni[muni.secs.apply(lambda S: ("cross_sector" in S) and (sector not in S))]
    has_specific=len(specific)>0
    has_direct=bool(specific.is_direct.any())
    funds=list(specific.opportunity_name.head(3))
    return dict(has_specific_route=has_specific, access="direct" if has_direct else ("competitive" if has_specific else "none"),
               n_specific=len(specific), n_broad=len(broad), example_funds=funds)
pd.DataFrame([{"sector":s, **supply_signal(s)} for s in actions.sector.unique()])

,sector,has_specific_route,access,n_specific,n_broad,example_funds
0,afolu,True,direct,5,7,"[Programa Concursable de Espacios Públicos, Pa..."
1,industry,False,none,0,9,[]
2,stationary_energy,True,direct,6,9,"[Comuna Energética, Asistencia Técnica para Mu..."
3,transportation,True,competitive,1,9,[Fondo de Infraestructura para el Desarrollo]
4,waste,True,direct,12,9,"[FPR 2026 - Fondo para el Reciclaje, FPR 2025 ..."


## Layer 3 — PROJECTS evidence: funded precedent (from `projects` + `project_funding`)

Has this kind of action been funded before, by what route, and at what size? This calibrates the read and provides the precedent annotation. It is evidence, not a forward input.

In [4]:
def precedent(action_id):
    pr=projects[projects.action_id==action_id]
    pids=set(pr.project_id)
    pf=project_funding[project_funding.project_id.isin(pids)]
    return dict(n_projects=len(pr),
                funding_routes=pf.source_label.value_counts().head(3).to_dict(),
                median_cost_MCLP=round(pd.to_numeric(pr.cost_total,errors="coerce").median(),0),
                median_duration_months=round(pd.to_numeric(pr.duration_months,errors="coerce").median(),0))
ex=actions.sort_values("bench_n_projects",ascending=False).iloc[0]["action_id"]
print("precedent for",ex,":",precedent(ex))

precedent for ipcc_0105 : {'n_projects': 152, 'funding_routes': {'SECTORIAL': 78, 'F.N.D.R.': 73, 'EMPRESA': 3}, 'median_cost_MCLP': np.float64(296.0), 'median_duration_months': np.float64(nan)}


/usr/local/lib/python3.10/dist-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


## Layer 4 — CITY: capacity and autonomy (external, from SINIM)

These two scores are **not** in finance_db; in production they come from the SINIM municipal review (one pair per comuna). Three illustrative profiles stand in here so the route logic can run.

In [5]:
cities=pd.DataFrame([
    ("Las Condes",0.85,0.80,"self-starter"),
    ("Iquique",   0.45,0.65,"capable-but-cash-tight"),
    ("Camiña",    0.15,0.20,"needs-full-support"),
], columns=["comuna","autonomy","capacity","archetype_city"])
cities

,comuna,autonomy,capacity,archetype_city
0,Las Condes,0.85,0.80,self-starter
1,Iquique,0.45,0.65,capable-but-cash-tight
2,Camiña,0.15,0.20,needs-full-support


## Putting it together — route bucket + financial_feasibility

Action demand is tested against the axis it loads on (capital vs autonomy, formulation vs capacity) to set a route bucket; the FINANCE signal then resolves the bucket's score (direct fund beats competitive beats no-usable-fund). This mirrors `methodology.md` §4 and §7.

In [6]:
def route_bucket(capital, formulation, autonomy, capacity):
    money_gap = capital > autonomy
    cap_gap   = formulation > capacity
    if capital <= 0.2:                 base="self-deliverable"
    elif not money_gap:                base="own-budget feasible"
    else:                              base="needs external co-finance"
    if cap_gap and base in ("self-deliverable","own-budget feasible"): base="needs technical assistance"
    elif cap_gap and base=="needs external co-finance":                base="needs external finance + TA / pooling"
    return base

SCORE={"self-deliverable":1.0,"own-budget feasible":0.85,"needs technical assistance":0.70}
def feasibility(bucket, access):
    if bucket in SCORE: return SCORE[bucket]
    base = {"direct":0.60,"competitive":0.45,"none":0.25}[access]   # co-finance: depends on supply
    return base-0.20 if "+ TA" in bucket else base                 # both gaps -> harder

def assess(action_row, city_row):
    sig=supply_signal(action_row.sector)
    bucket=route_bucket(action_row.capital_demand, action_row.formulation_demand, city_row.autonomy, city_row.capacity)
    score=round(feasibility(bucket, sig["access"]),2)
    return bucket, score, sig
print("logic ready")

logic ready


In [7]:
# run for a sector-spanning sample of actions x the three cities
sample=(actions.sort_values("bench_n_projects",ascending=False)
        .groupby("sector").head(1).head(5))
out=[]
for _,act in sample.iterrows():
    arow=a[a.action_id==act.action_id].iloc[0]
    for _,city in cities.iterrows():
        bucket,score,sig=assess(arow,city)
        out.append(dict(action=act.action_name[:34], sector=act.sector, comuna=city.comuna,
            route=bucket, financial_feasibility=score, access=sig["access"],
            named_fund=(sig["example_funds"][0][:30] if sig["example_funds"] else "—")))
pd.DataFrame(out)

,action,sector,comuna,route,financial_feasibility,access,named_fund
0,Promote active mobility through ro,transportation,Las Condes,own-budget feasible,0.85,competitive,Fondo de Infraestructura para
1,Promote active mobility through ro,transportation,Iquique,needs external finance + TA / pooling,0.25,competitive,Fondo de Infraestructura para
2,Promote active mobility through ro,transportation,Camiña,needs external finance + TA / pooling,0.25,competitive,Fondo de Infraestructura para
3,Deploy energy-efficient and solar-,stationary_energy,Las Condes,own-budget feasible,0.85,direct,Comuna Energética
4,Deploy energy-efficient and solar-,stationary_energy,Iquique,needs external finance + TA / pooling,0.40,direct,Comuna Energética
5,Deploy energy-efficient and solar-,stationary_energy,Camiña,needs external finance + TA / pooling,0.40,direct,Comuna Energética
6,Upgrade Landfills to Engineered Sa,waste,Las Condes,own-budget feasible,0.85,direct,FPR 2026 - Fondo para el Recic
7,Upgrade Landfills to Engineered Sa,waste,Iquique,needs external finance + TA / pooling,0.40,direct,FPR 2026 - Fondo para el Recic
8,Upgrade Landfills to Engineered Sa,waste,Camiña,needs external finance + TA / pooling,0.40,direct,FPR 2026 - Fondo para el Recic
9,Expand urban and peri-urban green,afolu,Las Condes,self-deliverable,1.00,direct,Programa Concursable de Espaci


## The annotation — the highest-value output

Per `methodology.md` §7 the score matters less than the *reason* attached. For one action and city, assemble the full annotation finance_db supports: the route, the named reachable fund, and the funded precedent with its benchmark.

In [8]:
arow=a[a.action_id==ex].iloc[0]; city=cities.iloc[2]  # needs-full-support comuna
bucket,score,sig=assess(arow,city); prec=precedent(ex)
print(f"Action  : {arow.action_name}  ({arow.sector})")
print(f"Comuna  : {city.comuna}  (autonomy {city.autonomy}, capacity {city.capacity})")
print(f"Route   : {bucket}   |   financial_feasibility = {score}")
print(f"Funds   : {sig['access']} access; e.g. {sig['example_funds'][:2]}")
print(f"Precedent: {prec['n_projects']} comparable funded projects, routes {prec['funding_routes']}")
print(f"Benchmark: ~{arow.bench_duration_median_months:.0f} months, ~{arow.bench_cost_median_MMCLP:,.0f} M CLP "
      f"(n={arow.bench_n_projects})")

Action  : Promote active mobility through road space reallocation  (transportation)
Comuna  : Camiña  (autonomy 0.15, capacity 0.2)
Route   : needs external finance + TA / pooling   |   financial_feasibility = 0.25
Funds   : competitive access; e.g. ['Fondo de Infraestructura para el Desarrollo']
Precedent: 152 comparable funded projects, routes {'SECTORIAL': 78, 'F.N.D.R.': 73, 'EMPRESA': 3}
Benchmark: ~nan months, ~296 M CLP (n=210)


/usr/local/lib/python3.10/dist-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


## What finance_db gives the methodology, and what it does not

- **Gives:** the ACTION demands (capital, formulation), the FINANCE supply + access-fit (which municipal funds, direct or gated), and the PROJECTS precedent + timeline/cost benchmark.
- **Needs externally:** the CITY autonomy/capacity (SINIM) — the only missing forward input here.
- **Still cannot:** adequacy (amount vs cost — BIP per-source amounts are blank) and award odds (no outcome data outside CONAF). These are the two facets `methodology.md` §0 marks out of scope, and the fixture confirms why.